In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import hashlib
import json

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AI/Project/TrafficGuard-AI"
)

assert PROJECT_ROOT.is_dir(), "Project not found"

required_files = [
    "src/trafficguard_engine.py",
    "src/trafficguard_refined.py",
    "src/video_processor.py",
    "configs/app_config.json",
    "models/helmet/helmet_best.pt",
    "models/seatbelt/seatbelt_best.pt",
    "models/context/vehicle_context_yolov8n.pt",
    "results/final_evaluation/tables/independent_test_metrics.csv"
]

missing = []

for relative in required_files:

    path = PROJECT_ROOT / relative

    if path.is_file():
        print("[FOUND]", relative)
    else:
        print("[MISSING]", relative)
        missing.append(relative)

assert not missing, missing


def sha256_file(path):

    sha = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024**2), b""
        ):
            sha.update(chunk)

    return sha.hexdigest()


ORIGINAL_HASHES = {
    relative: sha256_file(PROJECT_ROOT / relative)
    for relative in required_files
}

print("\nAll previous project assets verified.")
print("Ready for Phase 10.")

In [ ]:
from google.colab import files

import zipfile
import io
import tempfile
import shutil

uploaded = files.upload()

assert len(uploaded) == 1

filename, content = next(
    iter(uploaded.items())
)

assert filename.lower().endswith(".zip")

allowed_files = {
    "app/streamlit_app.py",
    "requirements_phase10.txt",
    ".streamlit/config.toml"
}

with zipfile.ZipFile(
    io.BytesIO(content)
) as archive:

    members = {
        p.filename
        for p in archive.infolist()
        if not p.is_dir()
    }

    assert members == allowed_files, (
        "Unexpected archive contents"
    )

    # Check every destination before writing.
    for relative in allowed_files:

        target = PROJECT_ROOT / relative

        if target.exists():

            assert target.read_bytes() == (
                archive.read(relative)
            ), f"Different existing file: {relative}"

    for relative in sorted(allowed_files):

        target = PROJECT_ROOT / relative

        target.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        if target.exists():

            print("[UNCHANGED]", relative)

        else:

            target.write_bytes(
                archive.read(relative)
            )

            print("[CREATED]", relative)

print("\nPHASE 10 APPLICATION INSTALLED.")
print("Original project files preserved.")

In [ ]:
import subprocess
import sys

REQUIREMENTS = (
    PROJECT_ROOT / "requirements_phase10.txt"
)

assert REQUIREMENTS.is_file()

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REQUIREMENTS)
    ],
    check=True
)

import streamlit
import ultralytics
import cv2
import torch
import pandas

print("Streamlit:", streamlit.__version__)
print("Ultralytics:", ultralytics.__version__)
print("OpenCV:", cv2.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

print("\nAPPLICATION DEPENDENCIES READY.")

In [ ]:
import py_compile

APP_PATH = (
    PROJECT_ROOT / "app/streamlit_app.py"
)

assert APP_PATH.is_file()

py_compile.compile(
    str(APP_PATH),
    doraise=True
)

config = json.loads(
    (
        PROJECT_ROOT /
        "configs/app_config.json"
    ).read_text()
)

for task, settings in config["models"].items():

    weights = (
        PROJECT_ROOT /
        settings["weights"]
    )

    assert weights.is_file()

    assert sha256_file(weights) == (
        settings["model_sha256"]
    )

    print(
        f"[VERIFIED] {task.upper()} MODEL"
    )

for relative, expected in ORIGINAL_HASHES.items():

    assert sha256_file(
        PROJECT_ROOT / relative
    ) == expected

print("\nApplication Python syntax: PASSED")
print("Original files unchanged: YES")
print("Model integrity: PASSED")

In [ ]:
import os
import subprocess
import time
import urllib.request

PORT = 8501

LOG_PATH = Path(
    "/content/trafficguard_streamlit.log"
)

env = os.environ.copy()

env["TRAFFICGUARD_ROOT"] = str(
    PROJECT_ROOT
)

log_file = open(
    LOG_PATH,
    "a"
)

streamlit_process = subprocess.Popen(

    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(APP_PATH),
        "--server.port",
        str(PORT),
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true"
    ],

    cwd=str(PROJECT_ROOT),

    env=env,

    stdout=log_file,

    stderr=subprocess.STDOUT

)

ready = False

for attempt in range(30):

    if streamlit_process.poll() is not None:

        break

    try:

        with urllib.request.urlopen(
            f"http://127.0.0.1:{PORT}/_stcore/health",
            timeout=2
        ) as response:

            if response.status == 200:

                ready = True
                break

    except Exception:

        time.sleep(1)

if ready:

    print("STREAMLIT SERVER RUNNING")
    print("Port:", PORT)
    print("Log:", LOG_PATH)

else:

    print(LOG_PATH.read_text()[-5000:])

    raise RuntimeError(
        "Streamlit server failed to start"
    )

In [ ]:
!cat /content/trafficguard_streamlit.log

In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8501)

In [ ]:
%pip install -q pyngrok

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

token = userdata.get(
    "NGROK_AUTHTOKEN"
)

assert token, "Ngrok token not configured"

ngrok.set_auth_token(token)

public_tunnel = ngrok.connect(
    addr=8501,
    proto="http"
)

print("=" * 60)
print("TRAFFICGUARD AI APPLICATION")
print("=" * 60)

print(public_tunnel.public_url)

print("\nOpen the URL in your browser.")

In [ ]:
import pandas as pd
import json

from datetime import datetime

PHASE10_RESULTS = (
    PROJECT_ROOT /
    "results/frontend_development"
)

PHASE10_RESULTS.mkdir(
    parents=True,
    exist_ok=True
)

frontend_files = [
    "app/streamlit_app.py",
    "requirements_phase10.txt",
    ".streamlit/config.toml"
]

verification = []

for relative in frontend_files:

    path = PROJECT_ROOT / relative

    verification.append({
        "File": relative,
        "Exists": path.is_file(),
        "Size KB": (
            round(path.stat().st_size / 1024, 2)
            if path.is_file()
            else 0
        )
    })

verification_df = pd.DataFrame(
    verification
)

display(verification_df)

assert verification_df["Exists"].all()

for relative, expected in ORIGINAL_HASHES.items():

    assert sha256_file(
        PROJECT_ROOT / relative
    ) == expected

print("Original models and source files preserved.")

report = {
    "project": "TrafficGuard AI",
    "phase": "10",
    "created_at": datetime.now().isoformat(),
    "frontend": "Streamlit",
    "source_files_verified": True,
    "original_files_unchanged": True,
    "training_performed": False,
    "default_profile": "standard",
    "implemented_pages": [
        "Overview",
        "Image detection",
        "Video detection",
        "CSV batch",
        "Model performance"
    ],
    "functional_acceptance": "Pending browser testing"
}

REPORT_PATH = (
    PHASE10_RESULTS /
    f"phase10_report_{datetime.now():%Y%m%d-%H%M%S-%f}.json"
)

REPORT_PATH.write_text(
    json.dumps(report, indent=4)
)

print(json.dumps(report, indent=4))

print("\nPHASE 10 FILE VERIFICATION PASSED")
print("Report:", REPORT_PATH)

In [ ]:
from pathlib import Path

LOG_PATH = Path(
    "/content/trafficguard_streamlit.log"
)

if LOG_PATH.is_file():

    print(
        LOG_PATH.read_text(
            errors="replace"
        )[-10000:]
    )

else:

    print("No Streamlit log found.")

In [ ]:
if "public_tunnel" in globals():

    from pyngrok import ngrok

    ngrok.disconnect(
        public_tunnel.public_url
    )

if "streamlit_process" in globals():

    streamlit_process.terminate()

    try:

        streamlit_process.wait(
            timeout=10
        )

    except subprocess.TimeoutExpired:

        streamlit_process.kill()

if "log_file" in globals():
    log_file.close()

print("TrafficGuard AI server stopped.")
print("All source files remain saved in Drive.")